In [1]:
# warning 무시
import warnings
warnings.filterwarnings("ignore")

import tensorflow as tf

# 새로운 초기 가중치로 처음부터 다시 학습
tf.keras.backend.clear_session()

# TensorFlow가 GPU 메모리를 처음부터 크게 선점하지 않고, 필요한 만큼 점진적으로 사용하도록 설정
gpus = tf.config.list_physical_devices("GPU")

for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

I0000 00:00:1788313930.263015  539550 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1788313930.627369  539550 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1788313932.511226  539550 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.resnet50 import decode_predictions
import numpy as np


In [ ]:
model = ResNet50(weights='imagenet') # 첫 실행 시 가중치 다운로드됨
 
# target_size=(224, 224) = 사진 크기를 가로 224 x 세로 224 픽셀로 맞춤
img = image.load_img('./images/YellowLabradorLooking_new.jpg',target_size=(224, 224))

# 결과 모양 (224,224,3) = 가로224 x 세로224 x 색상3개(빨강/초록/파랑, RGB)
x = image.img_to_array(img)    # x.shape=(224,224,3)

# 사진 앞에 '개수' 차원을 하나 더 붙입니다. (224,224,3) -> (1,224,224,3)
x = np.expand_dims(x, axis=0)  # x.shape=(1,244,244,3)

# 모델에게 이 사진이 무엇인지 예측(추론, predict)하게 시킵니다.
# 결과 pred = 1000종류 각각에 대한 '그럴 확률' 점수 (verbose=0 은 진행 로그를 숨기는 옵션)
pred = model.predict(x, verbose=0)

# 확률이 가장 높은 상위 3개(top=3)를 사람이 읽을 수 있는 이름으로 바꿔 출력합니다.
# decode_predictions = 숫자 결과를 'Labrador_retriever' 같은 실제 이름표로 번역해주는 함수
print('Predicted:', decode_predictions(pred, top=3))


I0000 00:00:1788313934.741911  539550 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5563 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9
I0000 00:00:1788313937.397769  542998 service.cc:153] XLA service 0x76173c064660 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1788313937.397807  542998 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9 (Driver: 12.7.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.25.1)
I0000 00:00:1788313937.445390  542998 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1788313937.908622  542998 cuda_dnn.cc:461] Loaded cuDNN version 92501


Predicted: [[('n02099712', 'Labrador_retriever', np.float32(0.26892528)), ('n02108089', 'boxer', np.float32(0.15150161)), ('n02099849', 'Chesapeake_Bay_retriever', np.float32(0.10756052))]]


I0000 00:00:1788313941.680091  542998 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


# 전이학습 + 미세조정

In [4]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten

# ResNet50을 가져오되, 이번엔 맨 위의 판정 부분 빼고 가져오기
resnet_model = ResNet50(input_shape=(224,224,3),include_top=False)

# 전이학습(Transfer Learning) + 전체 미세조정(Full Fine-tuning)
# trainable = True = 가져온 경력자의 기존 실력까지 포함해 '전부 다시 훈련 가능'하게 열어둠
resnet_model.trainable = True

# 깡통 모델 하나 만들고
model = Sequential()
# ResNet50 넣고
model.add(resnet_model)
model.add(Flatten())
model.add(Dense(1024, activation='relu')) # FC 층 추가
model.add(Dense(3, activation='softmax')) # 3개 클래스 분류
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 100352)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1024)           │   102,761,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │         3,075 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 126,352,259 (482.00 MB)

 Trainable params: 126,299,139 (481.79 MB)

 Non-trainable params: 53,120 (207.50 KB)